# CMA-ME — Baseline

**Method:** CMA-ME (Covariance Matrix Adaptation MAP-Elites)  
**Environments:** 4 Planar Arm settings (Det_Linear, Unc_Linear, Det_NonLinear, Unc_NonLinear)  
**Seeds:** 3 per environment  
**Task:** 1000-joint Planar Arm (CVT, 1950 bins)  
**Budget:** N_INIT_SAMPLES + N_EMITTERS × POPSIZE × N_STEPS

In [ ]:
# ============================================================
# Cell 1: Setup — mount Drive, clone repo, load SSLVE modules
# Run this cell once at the start of each Colab session.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!rm -rf org && git clone --filter=blob:none --sparse https://github.com/yugoguy/org.git
!cd org && git sparse-checkout set dev/SSLVE && git checkout Latent-Variable-Evolution
!pip install cma -q

import sys, os, glob
sys.path.insert(0, 'org/dev/SSLVE')
for f in sorted(glob.glob('org/dev/SSLVE/*.py')):
    %run {f}

print('Setup complete.')

Mounted at /content/drive
Cloning into 'org'...
remote: Enumerating objects: 1726, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 1726 (delta 140), reused 34 (delta 34), pack-reused 1501 (from 2)
Receiving objects: 100% (1726/1726), 484.56 KiB | 9.89 MiB/s, done.
Resolving deltas: 100% (692/692), done.
remote: Enumerating objects: 1, done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 1 (from 1)
Receiving objects: 100% (1/1), 255 bytes | 255.00 KiB/s, done.
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 18 (delta 4), reused 1 (delta 1), pack-reused 7 (from 1)
Receiving objects: 100% (18/18), 73.23 KiB | 5.63 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Branch 'Latent-Variable-Evolution' set up to track remote branch 'Latent-Variable-Evolution' from 'origin'.
Switched to a new branch 'Latent-Variable

In [ ]:
# ============================================================
# Cell 2: Method Hyperparameters
# ============================================================
import numpy as np
import random
import torch

# --- Architecture ---
N_JOINTS         = 1000   #@param {type:"integer"}
END_EFFECTOR_DIM = 2      #@param {type:"integer"}
N_BINS           = 1950   #@param {type:"integer"}
CENTERS          = "Precomputed_CVT_1950"  #@param ["Precomputed_CVT_1950", "CVT", "random"]
TOP_K            = 1      #@param {type:"integer"}
GREEDY_MEM       = True   #@param {type:"boolean"}

# --- CMA-ME ---
N_EMITTERS       = 10     #@param {type:"integer"}
SIGMA_INIT       = 0.075  #@param {type:"number"}
POPSIZE          = 50     #@param {type:"integer"}
N_INIT_SAMPLES   = 500    #@param {type:"integer"}
SEPARABLE        = True   #@param {type:"boolean"}

# --- Search budget ---
N_STEPS          = 1000   #@param {type:"integer"}

# --- Checkpoint base directory (saved to Google Drive) ---
CKPT_BASE_DIR    = '/content/drive/MyDrive/shared_ckpts/planar_arm/PlanarArm_CMAME/'  #@param {type:"string"}

total_evals = N_INIT_SAMPLES + N_EMITTERS * POPSIZE * N_STEPS
print(f'Method: CMA-ME')
print(f'N_JOINTS={N_JOINTS}, N_BINS={N_BINS}, TOP_K={TOP_K}')
print(f'N_EMITTERS={N_EMITTERS}, POPSIZE={POPSIZE}, SIGMA_INIT={SIGMA_INIT}')
print(f'N_INIT_SAMPLES={N_INIT_SAMPLES}, N_STEPS={N_STEPS}')
print(f'Total evaluations: {total_evals}')
print(f'Checkpoints will be saved to: {CKPT_BASE_DIR}')

Method: CMA-ME
N_JOINTS=1000, N_BINS=1950, TOP_K=1
N_EMITTERS=10, POPSIZE=50, SIGMA_INIT=0.075
N_INIT_SAMPLES=500, N_STEPS=1000
Total evaluations: 500500
Checkpoints will be saved to: /content/drive/MyDrive/shared_ckpts/planar_arm/PlanarArm_CMAME/


In [ ]:
# ============================================================
# Cell 3: Run All Experiments (4 envs x 3 seeds)
# Each run saves a checkpoint + plots immediately after finishing.
# ============================================================

ENV_CONFIGS = [
    {'name': 'Det_Linear',    'noise_sigma': 0.0,  'n_episodes': 1, 'fitness': 'angle_variance'},
    {'name': 'Unc_Linear',    'noise_sigma': 0.05, 'n_episodes': 3, 'fitness': 'angle_variance'},
    {'name': 'Det_NonLinear', 'noise_sigma': 0.0,  'n_episodes': 1, 'fitness': 'sine_dependency'},
    {'name': 'Unc_NonLinear', 'noise_sigma': 0.05, 'n_episodes': 3, 'fitness': 'sine_dependency'},
]

SEEDS = [42, 123, 456]

def _get_metric(info, key):
    v = info[key]
    return np.mean(v) if isinstance(v, list) else v

FITNESS_FNS = {
    'angle_variance':  lambda info: _get_metric(info, 'angle_variance'),
    'sine_dependency': lambda info: _get_metric(info, 'sine_dependency'),
}

all_results = []

ANGLES_PER_JOINT = END_EFFECTOR_DIM - 1
GENE_DIM = N_JOINTS * ANGLES_PER_JOINT
init_fn = lambda: np.random.uniform(-np.pi, np.pi, GENE_DIM)

for env_cfg in ENV_CONFIGS:
    env_name    = env_cfg['name']
    noise_sig   = env_cfg['noise_sigma']
    n_eps       = env_cfg['n_episodes']
    fitness_key = env_cfg['fitness']
    fitness_fn  = FITNESS_FNS[fitness_key]

    for seed in SEEDS:
        print(f'\n========== {env_name} | seed={seed} ==========')

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        collector = PlanarArmCollector(
            n_joints=N_JOINTS,
            end_effector_dim=END_EFFECTOR_DIM,
            noise_sigma=noise_sig,
            n_episodes=n_eps,
        )
        bd = PlanarArmBD_CVT(n_bins=N_BINS, centers=CENTERS, bd_dim=END_EFFECTOR_DIM)
        bm = MAPElitesBM(behavior_descriptor=bd, fitness_fn=fitness_fn, top_k=TOP_K, max_fitness=100)

        orchestrator = CMAME(
            agent_class=PlanarArmAgent,
            architecture=GENE_DIM,
            collector=collector,
            behavior_matching=bm,
            n_emitters=N_EMITTERS,
            sigma_init=SIGMA_INIT,
            popsize=POPSIZE,
            greedy_mem=GREEDY_MEM,
            n_init_samples=N_INIT_SAMPLES,
            init_fn=init_fn,
            separable=SEPARABLE,
        )

        orchestrator.run(n_steps=N_STEPS)

        # Save checkpoint immediately after this run
        ckpt_path = os.path.join(
            CKPT_BASE_DIR,
            f'CMAME_emit{N_EMITTERS}_pop{POPSIZE}_sigma{SIGMA_INIT}_{env_name}_seed{seed}/'
        )
        save_cmame_checkpoint(ckpt_path, orchestrator)
        print(f'Checkpoint saved: {ckpt_path}')

        # Save plots into checkpoint folder
        orchestrator.plot_history(save_path=ckpt_path + 'plot_history.png')
        print(f'Plots saved to: {ckpt_path}')

        # Force Drive sync after each run
        from google.colab import drive
        drive.flush_and_unmount()
        drive.mount('/content/drive')
        print('Drive re-synced.')

        # Print results
        f_min, f_mean, f_max = bm.fitness_stats()
        qd  = bm.qd_score()
        cov = bm.coverage()
        print(f'QD-score: {qd:.4f} | Coverage: {cov:.4f} | Best fitness: {f_min:.6f}')
        print(f'Total evaluations: {orchestrator.total_evals}')

        all_results.append({
            'env': env_name, 'seed': seed,
            'qd_score': qd, 'coverage': cov,
        })

print('\n====== All runs complete ======')

串流輸出內容已截斷至最後 5000 行。

--- CMAME Step 340/1000 ---
Archive: 34, Bins: 34, Coverage: 0.0174, Fitness min/mean/max: 1.90/3.68/7.34, QD-score: 3275.0240, Evals: 170000

--- CMAME Step 341/1000 ---
Archive: 34, Bins: 34, Coverage: 0.0174, Fitness min/mean/max: 1.89/3.67/7.34, QD-score: 3275.0779, Evals: 170500

--- CMAME Step 342/1000 ---
Archive: 34, Bins: 34, Coverage: 0.0174, Fitness min/mean/max: 1.88/3.67/7.34, QD-score: 3275.1447, Evals: 171000

--- CMAME Step 343/1000 ---
Archive: 34, Bins: 34, Coverage: 0.0174, Fitness min/mean/max: 1.87/3.67/7.34, QD-score: 3275.2131, Evals: 171500

--- CMAME Step 344/1000 ---
Archive: 34, Bins: 34, Coverage: 0.0174, Fitness min/mean/max: 1.87/3.67/7.34, QD-score: 3275.2479, Evals: 172000

--- CMAME Step 345/1000 ---
Archive: 34, Bins: 34, Coverage: 0.0174, Fitness min/mean/max: 1.87/3.65/7.34, QD-score: 3275.7384, Evals: 172500

--- CMAME Step 346/1000 ---
Archive: 34, Bins: 34, Coverage: 0.0174, Fitness min/mean/max: 1.87/3.64/7.34, QD-score: 327